In [ ]:
!pip install -q --no-deps --ignore-requires-python git+https://github.com/uiuc-focal-lab/syncode.git
!pip install -q fire interegular datasets jsonschema "transformers==4.53.2"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
syncode 0.4.16 requires regex==2024.11.6, but you have regex 2025.11.3 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import re
import torch
from syncode import Syncode

grammar = r"""
start: " " num " " op " " num "\n"
op: "+" | "-" | "*" | "/"
num: /-?[0-9]+(\.[0-9]*)?/
"""

syn_llm = Syncode(
    model="microsoft/phi-2",
    grammar=grammar,
    mode="grammar_strict",
    parse_output_only=True,
    max_new_tokens=30,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

[2026-10-26 07:33:26,712-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

[2026-10-26 07:34:49,738-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_1122482456_50257.pkl exist: False
[2026-10-26 07:34:49,739-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_1122482456_50257.pkl.
[2026-10-26 07:34:50,318-syncode.mask_store.fsm_set] - 7 FSMs with 18 states initialized in 0.00 seconds
[2026-10-26 07:34:50,319-syncode.mask_store.mask_store] - Ignore whitespace is False
[2026-10-26 07:34:50,330-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 49 to 11


100%|██████████| 18/18 [00:05<00:00,  3.58it/s]

[2026-10-26 07:34:55,400-syncode.mask_store.mask_store] - Time taken to create mask store: 5.66 seconds


In [ ]:
FEW_SHOT = """Extract the two operands and the operator (+, -, *, /) from the query.

Query: What is 40.5 plus 7.5?
Answer: 40.5 + 7.5

Query: What is 10.0 minus 2.25?
Answer: 10.0 - 2.25

Query: What is 9. multiplied by 4.5?
Answer: 9. * 4.5

Query: What is 65.0 divided by 2.0?
Answer: 65.0 / 2.0

"""

In [ ]:
OPS = {
    "+": lambda a, b: a + b,
    "-": lambda a, b: a - b,
    "*": lambda a, b: a * b,
    "/": lambda a, b: a / b if b != 0 else float("nan"),
}


def run_syncode(textual_query):
    prompt = FEW_SHOT + f"Query: {textual_query.strip()}\nAnswer:"
    output = syn_llm.infer(prompt)[0]
    m = re.search(r"(-?[0-9]+(?:\.[0-9]*)?) ([-+*/]) (-?[0-9]+(?:\.[0-9]*)?)", output)
    a, op, b = float(m.group(1)), m.group(2), float(m.group(3))
    answer = float(round(OPS[op](a, b), 10))
    return [a, b], op, answer

In [ ]:
tests = [
    ("What is 327. multiplied by 11.0?", [327.0, 11.0], "*", 3597.0),
    ("What is 45.1 plus 23.54?", [45.1, 23.54], "+", 68.64),
    ("What is 120.4 divided by 4.0?", [120.4, 4.0], "/", 30.1),
    ("What is 240.5 minus 30.25?", [240.5, 30.25], "-", 210.25),
    ("What is 4. plus 6.?", [4.0, 6.0], "+", 10.0),
    ("What is 2. multiplied by 8.0?", [2.0, 8.0], "*", 16.0),
    ("What is 33.3 divided by 3.33?", [33.3, 3.33], "/", 10.0),
    ("What is 1250.75 minus 10.5?", [1250.75, 10.5], "-", 1240.25),
    ("What is 5. multiplied by 6.?", [5.0, 6.0], "*", 30.0),
    ("What is -4.5 plus 10.0?", [-4.5, 10.0], "+", 5.5),
]

passed = 0
for q, ops, op, ans in tests:
    got = run_syncode(q)
    ok = got[0] == ops and got[1] == op and abs(got[2] - ans) < 1e-6
    passed += ok
    print("PASS" if ok else "FAIL", q, got)
print(f"{passed}/{len(tests)} passed")

PASS What is 327. multiplied by 11.0? ([327.0, 11.0], '*', 3597.0)
PASS What is 45.1 plus 23.54? ([45.1, 23.54], '+', 68.64)
PASS What is 120.4 divided by 4.0? ([120.4, 4.0], '/', 30.1)
PASS What is 240.5 minus 30.25? ([240.5, 30.25], '-', 210.25)
PASS What is 4. plus 6.? ([4.0, 6.0], '+', 10.0)
PASS What is 2. multiplied by 8.0? ([2.0, 8.0], '*', 16.0)
PASS What is 33.3 divided by 3.33? ([33.3, 3.33], '/', 10.0)
PASS What is 1250.75 minus 10.5? ([1250.75, 10.5], '-', 1240.25)
PASS What is 5. multiplied by 6.? ([5.0, 6.0], '*', 30.0)
PASS What is -4.5 plus 10.0? ([-4.5, 10.0], '+', 5.5)
10/10 passed
